# 🎬 PHA 2 (PART 3): DEEP LEARNING & PHỄU LỌC 3 TẦNG CHO ATD-12K TRÊN 2x GPU TESLA T4
## Sản Xuất Phân Mảnh Part 3 (`sakuga_golden_dataset_tps_part3.zip`) & Đảm Bảo Tính Liên Tục Chỉ Số Clip (`clip_004166+`)
**Đồ án PBL6 - Hệ Thống Nội Suy Khung Hình Anime Bằng Thin-Plate Spline & Deep Learning**

---

### ⚡ CÁC ĐẶC ĐIỂM NỔI BẬT CỦA NOTEBOOK PART 3:
1. **Đầu Vào Đơn Nhất & Tự Động Định Vị (Single Archive Input):**
   - Nhận trực tiếp file ZIP candidate ATD-12K (`atd12k_candidates_540p.zip` - 4,881 triplets Anime 2D `Japan_*` & `Hayao_*`).
   - Tự động quét và nhận diện file `dataset_manifest.json` trong `/kaggle/input` mà không cần cấu hình thủ công.
2. **Kế Thừa Chỉ Số Clip Chuẩn Xác (Canonical ID Continuity - Không Xung Đột):**
   - Part 1 & Part 2 trước đó đã tạo **4,165 Golden Triplets** (`clip_000001` đến `clip_004165`).
   - Notebook Part 3 tự động bắt đầu đánh số từ **`clip_004166`** $\to$ `clip_XXXXXX`. Tuyệt đối không ghi đè hoặc trùng ID với Part 1 và Part 2!
3. **Quy Trình Thực Thi Đơn Giản Trong 1 Bước (Single-Pass Flow):**
   - Với 4,881 candidate triplets, hệ thống chỉ cần chạy **1 đợt duy nhất** trên Dual GPU T4 (thời gian ước tính ~25–35 phút).
   - Không cần chia nhỏ thành nhiều batch và không cần thao tác dọn đĩa phức tạp giữa chừng.
4. **Đầu Ra Hoàn Chỉnh Để Tải Về & Ghép Nối:**
   - `sakuga_golden_dataset_tps_part3.zip`: Chứa toàn bộ các thư mục `train/clip_XXXXXX/`, `val/clip_XXXXXX/`, `matches/clip_XXXXXX/`.
   - `golden_meta_part3.json`: Metadata kỹ thuật chi tiết của Part 3.
   - `triplets_part3.csv` & `quality_audit_report_part3.csv`: Chỉ mục và báo cáo kiểm toán 12 chỉ số chuẩn vàng của Part 3.
   - Tự động gộp với `triplets_combined.csv` (nếu có trong Input) để xuất bản `triplets_combined_all.csv`.


---
### 🛠️ Bước 1: Kiểm Tra 2x GPU & Cài Đặt Môi Trường Thư Viện Deep Learning


In [ ]:
import os
import sys

# Khóa toàn bộ cảnh báo Python, PyTorch & OpenCV để triệt tiêu log đỏ trên giao diện Kaggle
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["OPENCV_LOG_LEVEL"] = "OFF"
import warnings
warnings.filterwarnings("ignore")

!nvidia-smi

# Cài đặt các thư viện cần thiết tương thích với PyTorch & CUDA trên Kaggle
!pip install -q timm gdown einops kornia omegaconf

import glob
import json
import time
import shutil
import random
import zipfile
import gc
import csv
import types
from collections import defaultdict
import cv2

# Khóa log OpenCV C++ an toàn để tránh in cảnh báo ra stderr trên mọi phiên bản OpenCV
try:
    if hasattr(cv2, 'utils') and hasattr(cv2.utils, 'logging'):
        cv2.utils.logging.setLogLevel(cv2.utils.logging.LOG_LEVEL_SILENT)
    elif hasattr(cv2, 'setLogLevel'):
        cv2.setLogLevel(0)
except Exception:
    pass

# Giới hạn OpenCV 1 thread cho mỗi worker để chống nghẽn CPU & deadlock OpenMP khi chạy song song 2 GPU
cv2.setNumThreads(1)

import torch
import torch.nn.functional as F
import numpy as np
from scipy import ndimage
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

print("=" * 75)
print(f"[*] Python Version  : {sys.version.split()[0]}")
print(f"[*] PyTorch Version : {torch.__version__}")
print(f"[*] OpenCV Version  : {cv2.__version__}")
num_gpus = torch.cuda.device_count()
print(f"[*] Số lượng GPU    : {num_gpus}")
if torch.cuda.is_available():
    for i in range(num_gpus):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"    + GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.2f} GB VRAM)")
print("=" * 75)

# Clone các repository mã nguồn Deep Learning
if not os.path.exists("GlueStick"):
    print("[*] Đang clone mã nguồn GlueStick...")
    !git clone --depth 1 https://github.com/cvg/GlueStick.git
!pip install -q --no-deps -e ./GlueStick

if not os.path.exists("AniLines"):
    print("[*] Đang clone mã nguồn AniLines...")
    !git clone --depth 1 https://github.com/zhenglinpan/AniLines-Anime-Lineart-Extractor.git AniLines

if not os.path.exists("anime_seg"):
    print("[*] Đang clone mã nguồn anime_seg...")
    !git clone --depth 1 https://github.com/SkyTNT/anime-segmentation.git anime_seg

for p in ["GlueStick", "/kaggle/working/GlueStick", "AniLines", "/kaggle/working/AniLines", "anime_seg", "anime_seg/model"]:
    p_abs = os.path.abspath(p)
    if os.path.exists(p_abs) and p_abs not in sys.path:
        sys.path.insert(0, p_abs)

# Mock pytlsd bằng OpenCV LSD native C++ an toàn, chống rò rỉ RAM và chống vô tận vòng lặp CPU
def cv2_lsd(img, scale=1.0):
    if not isinstance(img, np.ndarray):
        img = np.array(img)
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if img.dtype != np.uint8:
        if img.max() <= 1.01 and img.max() > 0:
            img = (img * 255.0).clip(0, 255).astype(np.uint8)
        else:
            img = np.clip(img, 0, 255).astype(np.uint8)
    if scale != 1.0:
        h, w = img.shape[:2]
        img_s = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    else:
        img_s = img
    detector = cv2.createLineSegmentDetector(cv2.LSD_REFINE_STD)
    lines, _, _, _ = detector.detect(img_s)
    if lines is None or len(lines) == 0:
        return np.zeros((0, 5), dtype=np.float32)
    lines = lines.reshape(-1, 4).astype(np.float32)
    if scale != 1.0:
        lines /= float(scale)
    scores = np.ones((len(lines), 1), dtype=np.float32)
    return np.hstack([lines, scores])

fake_pytlsd = types.ModuleType("pytlsd")
fake_pytlsd.lsd = cv2_lsd
sys.modules["pytlsd"] = fake_pytlsd

try:
    import gluestick.models.wireframe as wireframe_mod
    wireframe_mod.lsd = cv2_lsd
except Exception:
    pass

# Nạp trước và kiểm tra tính sẵn sàng của GlueStick TwoViewPipeline ngay tại Bước 1
try:
    from gluestick.models.two_view_pipeline import TwoViewPipeline
    print("[✓] GlueStick TwoViewPipeline đã import thành công tại Bước 1!")
except Exception as e_gs:
    try:
        from model.gluestick.models.two_view_pipeline import TwoViewPipeline
        print("[✓] Fallback model.gluestick.models.two_view_pipeline thành công tại Bước 1!")
    except Exception as e_gs2:
        print(f"[!] Cảnh báo import GlueStick tại Bước 1: {e_gs2}")

print("[✓] Môi trường và cơ chế Mock LSD chống rò rỉ RAM đã sẵn sàng!")


---
### 📥 Bước 2: Tự Động Tải Trọng Số Pretrained (GlueStick, ISNetDIS, AniLines Detail)


In [ ]:
os.makedirs("weights", exist_ok=True)

# 1. Trọng số GlueStick MD Checkpoint
gluestick_ckpt = "weights/checkpoint_GlueStick_MD.tar"
if not os.path.exists(gluestick_ckpt) or os.path.getsize(gluestick_ckpt) < 1000000:
    print("[*] Đang tải GlueStick checkpoint...")
    url_gs = "https://github.com/cvg/GlueStick/releases/download/v0.1_arxiv/checkpoint_GlueStick_MD.tar"
    !wget -q -O {gluestick_ckpt} {url_gs}
    print("[✓] GlueStick weights tải xong!")

# 2. Trọng số ISNetDIS (Anime Background Segmentation)
isnet_ckpt = "weights/isnetis.ckpt"
if not os.path.exists(isnet_ckpt) or os.path.getsize(isnet_ckpt) < 1000000:
    print("[*] Đang tải ISNetDIS checkpoint...")
    url_isnet = "https://huggingface.co/skytnt/anime-seg/resolve/main/isnetis.ckpt"
    !wget -q -O {isnet_ckpt} {url_isnet}
    print("[✓] ISNetDIS weights tải xong!")

# 3. Trọng số AniLines Detail
anilines_ckpt = "weights/detail.pth"
if not os.path.exists(anilines_ckpt) or os.path.getsize(anilines_ckpt) < 1000000:
    print("[*] Đang tải AniLines Detail checkpoint...")
    !gdown --id 12U1Mwlonoipk2Yvr12mNaFB30foy420o -O {anilines_ckpt} --fuzzy
    print("[✓] AniLines Detail weights tải xong!")

print()
print("=" * 60)
print("[✓] Danh mục trọng số Deep Learning trong thư mục weights/:")
!ls -lh weights/
print("=" * 60)


---
### 🧠 Bước 3: Khởi Tạo Lớp Dual-GPU Pipeline Worker (Hỗ Trợ 2x GPU Song Song & GPU Batching)


In [ ]:
# Đảm bảo đường dẫn mã nguồn model luôn có mặt trong sys.path
for p in ["GlueStick", "/kaggle/working/GlueStick", "AniLines", "/kaggle/working/AniLines", "anime_seg", "anime_seg/model"]:
    p_abs = os.path.abspath(p)
    if os.path.exists(p_abs) and p_abs not in sys.path:
        sys.path.insert(0, p_abs)

try:
    from gluestick.models.two_view_pipeline import TwoViewPipeline
except ImportError:
    try:
        from model.gluestick.models.two_view_pipeline import TwoViewPipeline
    except ImportError:
        try:
            from gluestick import get_model
            TwoViewPipeline = get_model('two_view_pipeline')
        except Exception:
            TwoViewPipeline = None

try:
    from network.line_extractor import LineExtractor
except ImportError:
    try:
        from AniLines.network.line_extractor import LineExtractor
    except ImportError:
        from model.line_extractor import LineExtractor

try:
    from anime_seg.model.isnet import ISNetDIS
except ImportError:
    try:
        from model.isnet import ISNetDIS
    except ImportError:
        from isnet import ISNetDIS

def load_gluestick(device, weights_path="weights/checkpoint_GlueStick_MD.tar"):
    global TwoViewPipeline
    if 'TwoViewPipeline' not in globals() or TwoViewPipeline is None:
        for p in ["GlueStick", "/kaggle/working/GlueStick"]:
            p_abs = os.path.abspath(p)
            if os.path.exists(p_abs) and p_abs not in sys.path:
                sys.path.insert(0, p_abs)
        try:
            from gluestick.models.two_view_pipeline import TwoViewPipeline
        except ImportError:
            try:
                from model.gluestick.models.two_view_pipeline import TwoViewPipeline
            except ImportError:
                from gluestick import get_model
                TwoViewPipeline = get_model('two_view_pipeline')

    conf = {
        'name': 'two_view_pipeline',
        'use_lines': True,
        'extractor': {
            'name': 'wireframe',
            'sp_params': {'force_num_keypoints': False, 'max_num_keypoints': 1000},
            'wireframe_params': {'merge_points': True, 'merge_line_endpoints': True},
            'max_n_lines': 300,
        },
        'matcher': {'name': 'gluestick', 'weights': weights_path, 'trainable': False},
        'ground_truth': {'from_pose_depth': False}
    }
    model = TwoViewPipeline(conf).to(device)
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return model

class AnimeBgRemover:
    def __init__(self, device, ckpt_path="weights/isnetis.ckpt", img_size=1024):
        global ISNetDIS
        if 'ISNetDIS' not in globals() or ISNetDIS is None:
            try:
                from anime_seg.model.isnet import ISNetDIS
            except ImportError:
                try:
                    from model.isnet import ISNetDIS
                except ImportError:
                    from isnet import ISNetDIS
        self.device = device
        self.img_size = img_size
        self.net = ISNetDIS().to(self.device)
        try:
            ckpt = torch.load(ckpt_path, map_location=self.device, weights_only=False)
        except TypeError:
            ckpt = torch.load(ckpt_path, map_location=self.device)
        sd = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt
        clean_sd = {k.replace('net.', ''): v for k, v in sd.items() if k.startswith('net.')}
        if not clean_sd:
            clean_sd = sd
        self.net.load_state_dict(clean_sd)
        self.net.eval()
        for p in self.net.parameters():
            p.requires_grad = False

    def remove_background_batch(self, img_bgr_list, batch_size=16):
        """
        Xử lý tách nền theo Batch trên GPU cho danh sách ảnh BGR 540p (batch_size=16).
        Tối ưu hóa:
        - Chuyển toàn bộ ảnh uint8 lên GPU, float/255.0 và resize + pad trực tiếp trên GPU.
        - Trộn nền trắng và tính fg_ratios trực tiếp trên GPU bằng PyTorch tensor.
        - Trả về danh sách: [(white_bg_uint8, fg_mask_bool, float(fg_ratio)), ...]
        """
        results = []
        n_imgs = len(img_bgr_list)
        if n_imgs == 0:
            return results

        s = self.img_size
        for start_idx in range(0, n_imgs, batch_size):
            sub_imgs = img_bgr_list[start_idx : start_idx + batch_size]
            b_size = len(sub_imgs)

            h0, w0 = sub_imgs[0].shape[:2]
            h, w = (s, int(s * w0 / h0)) if h0 > w0 else (int(s * h0 / w0), s)
            ph, pw = s - h, s - w
            pad_left = pw // 2
            pad_right = pw - pad_left
            pad_top = ph // 2
            pad_bottom = ph - pad_top

            # Chuyển batch uint8 lên GPU -> permute -> float / 255.0 trên GPU
            stack_u8 = np.stack(sub_imgs, axis=0)
            orig_t = torch.from_numpy(stack_u8).to(self.device, non_blocking=True).permute(0, 3, 1, 2).float() / 255.0

            # Resize & Pad trực tiếp trên GPU
            resized_t = F.interpolate(orig_t, size=(h, w), mode='bilinear', align_corners=False)
            inp_tensor = F.pad(resized_t, (pad_left, pad_right, pad_top, pad_bottom))

            with torch.no_grad():
                if 'cuda' in self.device.type:
                    with torch.amp.autocast('cuda'):
                        out = self.net(inp_tensor)
                else:
                    out = self.net(inp_tensor)

                # ISNetDIS trả về tuple ([d1, d2, d3, d4, d5, d6], [features...])
                if isinstance(out, (list, tuple)):
                    first = out[0]
                    raw_pred = first[0] if isinstance(first, (list, tuple)) else first
                else:
                    raw_pred = out

                pred = raw_pred.sigmoid() if hasattr(raw_pred, 'sigmoid') else torch.sigmoid(raw_pred)

                if pred.ndim == 3:
                    pred = pred.unsqueeze(1)

                pred_crop = pred[:, :, pad_top : pad_top + h, pad_left : pad_left + w]
                masks_t = F.interpolate(pred_crop.float(), size=(h0, w0), mode='bilinear', align_corners=False)

                # Tính fg_ratios trực tiếp trên GPU
                fg_ratios = (masks_t > 0.5).float().mean(dim=(1, 2, 3)).cpu().numpy()

                # Trộn nền trắng trực tiếp trên GPU
                white_t = torch.clamp(masks_t * orig_t * 255.0 + (1.0 - masks_t) * 255.0, 0, 255).to(torch.uint8)
                white_bgr_np = white_t.permute(0, 2, 3, 1).contiguous().cpu().numpy()
                fg_masks_np = (masks_t.squeeze(1) > 0.5).cpu().numpy()

            for b_i in range(b_size):
                results.append((white_bgr_np[b_i], fg_masks_np[b_i], float(fg_ratios[b_i])))

            del orig_t, resized_t, inp_tensor, out, raw_pred, pred, pred_crop, masks_t, white_t

        return results

    def remove_background(self, img_bgr):
        return self.remove_background_batch([img_bgr], batch_size=1)[0][0]

def load_line_extractor(device, weights_path="weights/detail.pth"):
    global LineExtractor
    if 'LineExtractor' not in globals() or LineExtractor is None:
        try:
            from network.line_extractor import LineExtractor
        except ImportError:
            try:
                from AniLines.network.line_extractor import LineExtractor
            except ImportError:
                from model.line_extractor import LineExtractor
    extractor = LineExtractor(2, 1, True).to(device)
    try:
        sd_line = torch.load(weights_path, map_location=device, weights_only=True)
    except TypeError:
        sd_line = torch.load(weights_path, map_location=device)
    extractor.load_state_dict(sd_line)
    extractor.eval()
    for p in extractor.parameters():
        p.requires_grad = False
    return extractor

# Lớp Worker đại diện cho 1 GPU cụ thể (chứa trọn gói 3 model trên GPU đó)
class DualGpuPipelineWorker:
    def __init__(self, device_id):
        device_str = f"cuda:{device_id}" if torch.cuda.is_available() else "cpu"
        self.device = torch.device(device_str)
        self.device_id = device_id
        print(f"[*] Đang khởi tạo DualGpuPipelineWorker trên: {self.device}...")
        self.gluestick = load_gluestick(self.device)
        self.bg_remover = AnimeBgRemover(self.device)
        self.line_extractor = load_line_extractor(self.device)

        # Sobel kernels trên GPU cho AniLines input
        sobel_x = torch.tensor([[-1.0, 0.0, 1.0],
                                [-2.0, 0.0, 2.0],
                                [-1.0, 0.0, 1.0]], dtype=torch.float32, device=self.device).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1.0, -2.0, -1.0],
                                [ 0.0,  0.0,  0.0],
                                [ 1.0,  2.0,  1.0]], dtype=torch.float32, device=self.device).view(1, 1, 3, 3)
        self.sobel_x = sobel_x
        self.sobel_y = sobel_y
        self.bgr_weights = torch.tensor([0.114, 0.587, 0.299], dtype=torch.float32, device=self.device).view(1, 3, 1, 1)

        print(f"[✓] Worker trên {self.device} đã sẵn sàng!")

    def remove_background_batch(self, img_bgr_list, batch_size=16):
        return self.bg_remover.remove_background_batch(img_bgr_list, batch_size=batch_size)

    def remove_background(self, img_bgr):
        return self.bg_remover.remove_background(img_bgr)

    @torch.no_grad()
    def extract_lineart_and_edt_batch(self, white_imgs_list, batch_size=16):
        """
        Trích xuất Lineart bằng AniLines theo Batch trên GPU (batch_size=16),
        làm mịn nét trực tiếp trên CUDA, tính EDT (swdr) tức thì qua F.max_pool2d trên GPU,
        và tích hợp tính luôn Canvas Purity (pwcr, rgd, density) trực tiếp trên GPU.
        Trả về: [(line_uint8, float(swdr), {'pwcr': float, 'rgd': float, 'density': float}), ...]
        """
        results = []
        n_imgs = len(white_imgs_list)
        if n_imgs == 0:
            return results

        for start_idx in range(0, n_imgs, batch_size):
            sub_imgs = white_imgs_list[start_idx : start_idx + batch_size]
            b_size = len(sub_imgs)
            h0, w0 = sub_imgs[0].shape[:2]

            # [B, H, W, 3] uint8 -> [B, 3, H, W] float [0, 1] trên GPU
            stack_np = np.stack(sub_imgs, axis=0)
            t_bgr = torch.from_numpy(stack_np).to(self.device, non_blocking=True).permute(0, 3, 1, 2).float() / 255.0

            # Grayscale trên GPU
            grays = torch.sum(t_bgr * self.bgr_weights, dim=1, keepdim=True)

            # Sobel magnitude trên GPU
            pad = F.pad(grays, (1, 1, 1, 1), mode='replicate')
            gx = F.conv2d(pad, self.sobel_x)
            gy = F.conv2d(pad, self.sobel_y)
            mag = torch.sqrt(gx * gx + gy * gy + 1e-8)

            mag_min = mag.amin(dim=(2, 3), keepdim=True)
            mag_max = mag.amax(dim=(2, 3), keepdim=True)
            sobel_norm = 1.0 - (mag - mag_min) / (mag_max - mag_min + 1e-6)

            # Input vào AniLines: [B, 2, H, W]
            x_in = torch.cat([grays, sobel_norm], dim=1)

            pad_h = 8 - (h0 % 8) if (h0 % 8) != 0 else 0
            pad_w = 8 - (w0 % 8) if (w0 % 8) != 0 else 0
            if pad_h > 0 or pad_w > 0:
                x_in = F.pad(x_in, (0, pad_w, 0, pad_h), mode='reflect')

            # Bọc lời gọi self.line_extractor trong torch.amp.autocast('cuda') và ép .float()
            if 'cuda' in self.device.type:
                with torch.amp.autocast('cuda'):
                    pred_line = self.line_extractor(x_in)[:, :, :h0, :w0].float()
            else:
                pred_line = self.line_extractor(x_in)[:, :, :h0, :w0].float()

            # Làm mịn nét trên GPU (GPU Tensor Smoothing)
            try:
                import kornia.filters as kf
                pred_smooth = kf.bilateral_blur(pred_line, (5, 5), 0.15, (1.5, 1.5))
            except Exception:
                g_kernel = torch.tensor([[1., 2., 1.],
                                         [2., 4., 2.],
                                         [1., 2., 1.]], dtype=torch.float32, device=self.device).view(1, 1, 3, 3) / 16.0
                pad_line = F.pad(pred_line, (1, 1, 1, 1), mode='replicate')
                pred_smooth = F.conv2d(pad_line, g_kernel)

            # TÍNH TOÁN EDT (swdr) TỨC THÌ TRÊN GPU BẰNG F.max_pool2d
            stroke_mask = (pred_smooth < (180.0 / 255.0)).float()
            inv_stroke = 1.0 - stroke_mask
            pool_inv = F.max_pool2d(inv_stroke, kernel_size=7, stride=1, padding=3)
            thick_mask = (stroke_mask == 1.0) & (pool_inv == 0.0)

            total_strokes = stroke_mask.sum(dim=(2, 3)).squeeze(1).cpu().numpy()
            thick_strokes = thick_mask.sum(dim=(2, 3)).squeeze(1).cpu().numpy()

            # TÍNH TOÁN CANVAS PURITY TRỰC TIẾP TRÊN GPU TỪ pred_smooth
            lines_u8 = torch.clamp(pred_smooth.squeeze(1) * 255.0, 0, 255).round()
            pwcr_arr = (lines_u8 >= 250).float().mean(dim=(1, 2)).cpu().numpy()
            rgd_arr = ((lines_u8 >= 180) & (lines_u8 < 250)).float().mean(dim=(1, 2)).cpu().numpy()
            density_arr = (lines_u8 < 180).float().mean(dim=(1, 2)).cpu().numpy()

            lines_uint8 = lines_u8.to(torch.uint8).cpu().numpy()

            for b_i in range(b_size):
                tot_s = float(total_strokes[b_i])
                thk_s = float(thick_strokes[b_i])
                swdr = (thk_s / max(tot_s, 1.0)) if tot_s > 0 else 0.0
                cp_dict = {
                    'pwcr': float(pwcr_arr[b_i]),
                    'rgd': float(rgd_arr[b_i]),
                    'density': float(density_arr[b_i])
                }
                results.append((lines_uint8[b_i], float(swdr), cp_dict))

            del t_bgr, grays, pad, gx, gy, mag, sobel_norm, x_in, pred_line, pred_smooth, stroke_mask, inv_stroke, pool_inv, thick_mask, lines_u8

        return results

    def extract_lineart_smooth(self, img_white):
        return self.extract_lineart_and_edt_batch([img_white], batch_size=1)[0][0]

    @torch.no_grad()
    def extract_gluestick_feats(self, gray_uint8):
        """
        Trích xuất đặc trưng GlueStick (points & lines) cho 1 frame đơn lẻ.
        Chống việc chạy lại cv2_lsd / SuperPoint trên CPU nhiều lần cho cùng 1 frame.
        """
        t_img = torch.from_numpy(gray_uint8).float().to(self.device)[None, None] / 255.0
        if hasattr(self.gluestick, 'extract_view'):
            feats = self.gluestick.extract_view({'image0': t_img}, '0')
        elif hasattr(self.gluestick, 'extractor'):
            feats = self.gluestick.extractor({'image': t_img})
        else:
            raw = self.gluestick({'image0': t_img, 'image1': t_img})
            feats = {k[:-1]: v for k, v in raw.items() if k.endswith('0')}
        return feats

    @torch.no_grad()
    def match_gluestick_pair(self, feats0, feats1, shape_hw):
        """
        Khớp cặp đặc trưng GlueStick đã trích xuất sẵn giữa 2 frame trên GPU ở chế độ FP32.
        Tắt autocast FP16 tại bước matcher để tránh lỗi 'scatter(): Expected self.dtype to be equal to src.dtype'
        bên trong mạng GNN của GlueStick.
        """
        h, w = shape_hw
        dummy_0 = torch.empty((1, 1, h, w), dtype=torch.float32, device=self.device)
        dummy_1 = torch.empty((1, 1, h, w), dtype=torch.float32, device=self.device)
        data = {'image0': dummy_0, 'image1': dummy_1}

        pred0 = {k if k.endswith('0') else k + '0': (v.float() if torch.is_floating_point(v) else v) for k, v in feats0.items()}
        pred1 = {k if k.endswith('1') else k + '1': (v.float() if torch.is_floating_point(v) else v) for k, v in feats1.items()}
        matcher_input = {**data, **pred0, **pred1}

        if 'cuda' in self.device.type:
            with torch.amp.autocast('cuda', enabled=False):
                match_out = self.gluestick.matcher(matcher_input)
        else:
            match_out = self.gluestick.matcher(matcher_input)

        res = {**matcher_input, **match_out}
        return res

num_devices = max(1, torch.cuda.device_count())
print(f"[*] Khởi tạo {num_devices} Pipeline Workers song song...")
gpu_workers = [DualGpuPipelineWorker(i) for i in range(num_devices)]
print(f"[✓] Đã khởi tạo thành công {len(gpu_workers)} Workers trên tất cả các GPU!")


---
### ⚖️ Bước 4: Module Phễu Lọc 3 Cửa Sổ Chuẩn Vàng (Quality Metrics Engine - 12 Chỉ Số)


In [ ]:
THRESHOLDS = {
    # Nhóm A: Motion Dynamics & Kinematics (Cửa Sổ 3)
    "M1_DSMD_MIN": 1.50,       # px: Vận tốc tối thiểu cả 2 chặng (loại bỏ đứng im)
    "M1_DSMD_MAX": 14.0,       # px: Vận tốc tối đa cho phép
    "M2_MBR_MIN": 0.50,        # Tỷ lệ cân bằng vận tốc 2 chặng >= 0.50 (nới nhẹ để chấp nhận ease-in/ease-out)
    "M3_SSPR_MAX": 0.30,       # Nét đứng yên trùng khít cả 3 frame <= 30%
    "M4_MDCC_MIN": 0.35,       # Nhất quán góc chuyển động >= 0.35 (chấp nhận gia tốc tự nhiên)
    
    # Nhóm B: Canvas Purity & Background Isolation (Cửa Sổ 1)
    "B1_PWCR_MIN": 0.85,       # Tỷ lệ nền trắng tinh (>= 250) trên ảnh Lineart >= 85%
    "B2_RGD_MAX": 0.065,       # Tỷ lệ pixel xám dơ [180, 250) trên ảnh Lineart <= 6.5%
    "B3_BSPI_MAX": 0.15,       # Chỉ số đột biến mật độ nét giữa frame <= 15% (nới từ 6% cho Sakuga)
    "B4_BTV_MAX": 4.0,         # Phương sai cường độ sáng nền tĩnh <= 4.0
    
    # Nhóm C: Stroke Topology & Continuity (Cửa Sổ 2)
    "C1_DCCR_MIN": 0.55,       # Tỷ lệ diện tích cụm nét chính sau dilation >= 55% (hiệu chỉnh thực tế)
    "C2_MCCC_MAX": 250,        # Tổng số cụm nét rời rạc <= 250 cụm (cho phép chi tiết mắt, miệng, nếp áo)
    "C3_TNFAR_MAX": 0.015,     # Tỷ lệ bụi rác li ti (< 15px) <= 1.5%
    "C4_WSDS_MIN": 0.18,       # Warped Stroke Dice Score >= 0.18 (hiệu chuẩn nét hairline)
    
    # Nhóm D: VFX & Stroke Thickness (Cửa Sổ 2)
    "V1_SWDR_MAX": 0.04,       # Tỷ lệ nét dày quá khổ (EDT > 3px, nét > 6px) <= 4.0%
}

DARK_THRESH = 180              # Pixel < 180 được tính là nét vẽ
WHITE_THRESH = 250             # Pixel >= 250 là nền trắng tuyệt đối

def calc_canvas_purity(gray_line_img):
    """
    Đo độ tinh khiết canvas TRÊN ẢNH LINEART (sau AniLines).
    Loại bỏ triệt để lỗi đo nhầm màu da/quần áo sáng thành nền xám dơ.
    """
    h, w = gray_line_img.shape[:2]
    total_pixels = float(h * w)
    pure_white_count = np.sum(gray_line_img >= WHITE_THRESH)
    pwcr = float(pure_white_count) / total_pixels
    gray_contam_count = np.sum((gray_line_img >= DARK_THRESH) & (gray_line_img < WHITE_THRESH))
    rgd = float(gray_contam_count) / total_pixels
    stroke_count = np.sum(gray_line_img < DARK_THRESH)
    density = float(stroke_count) / total_pixels
    return {'pwcr': pwcr, 'rgd': rgd, 'density': density}

def calc_bspi(dens0, dens_gt, dens1):
    avg_01 = (dens0 + dens1) / 2.0
    delta_gt = abs(dens_gt - avg_01) / max(avg_01, 1e-6)
    delta_01 = abs(dens0 - dens1) / max(max(dens0, dens1), 1e-6)
    bspi = max(delta_gt, delta_01)
    return bspi, delta_gt, delta_01

def calc_stroke_topology(gray_line_img):
    """
    Thẩm định hình thái nét vẽ:
    - mccc & tnfar tính trên stroke_mask gốc để phát hiện bụi rác.
    - Early-Exit: nếu mccc > C2_MCCC_MAX hoặc tnfar > C3_TNFAR_MAX thì trả về ngay,
      không chạy cv2.dilate và lần connectedComponents thứ 2.
    - dccr áp dụng dilation 7x7 để kết nối các nét vẽ của cùng một nhân vật.
    """
    stroke_mask = (gray_line_img < DARK_THRESH).astype(np.uint8) if gray_line_img.dtype != np.uint8 else (gray_line_img < DARK_THRESH).view(np.uint8)
    if stroke_mask.dtype != np.uint8:
        stroke_mask = stroke_mask.astype(np.uint8)
    total_stroke_pixels = np.sum(stroke_mask)
    if total_stroke_pixels == 0:
        return {'dccr': 0.0, 'mccc': 0, 'tnfar': 1.0}

    # 1. mccc và tnfar trên nét vẽ gốc
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(stroke_mask, connectivity=8)
    if num_labels <= 1:
        return {'dccr': 0.0, 'mccc': 0, 'tnfar': 1.0}
    areas = stats[1:, cv2.CC_STAT_AREA]
    mccc = len(areas)
    tiny_pixels = np.sum(areas[areas < 15])
    tnfar = float(tiny_pixels) / float(total_stroke_pixels)

    # Early-Exit: Nếu trượt mccc hoặc tnfar, thoát ngay lập tức mà KHÔNG chạy cv2.dilate
    if mccc > THRESHOLDS['C2_MCCC_MAX'] or tnfar > THRESHOLDS['C3_TNFAR_MAX']:
        return {'dccr': 0.0, 'mccc': mccc, 'tnfar': tnfar}

    # 2. dccr: Giãn nở hình thái học để kết nối các nét vẽ rời rạc của cùng một nhân vật
    dilated_mask = cv2.dilate(stroke_mask, np.ones((7, 7), np.uint8), iterations=1)
    num_d, _, stats_d, _ = cv2.connectedComponentsWithStats(dilated_mask, connectivity=8)
    if num_d > 1:
        areas_d = stats_d[1:, cv2.CC_STAT_AREA]
        total_dilated = np.sum(areas_d)
        max_dilated = np.max(areas_d)
        dccr = float(max_dilated) / float(max(total_dilated, 1))
    else:
        dccr = 0.0

    return {'dccr': dccr, 'mccc': mccc, 'tnfar': tnfar}

def calc_stroke_width_edt(gray_line_img):
    stroke_mask = (gray_line_img < DARK_THRESH).astype(np.uint8)
    total_strokes = np.sum(stroke_mask)
    if total_strokes == 0:
        return {'swdr': 0.0}
    edt = ndimage.distance_transform_edt(stroke_mask)
    thick_pixels = np.sum(edt > 3.0)
    swdr = float(thick_pixels) / float(total_strokes)
    return {'swdr': swdr}

def calc_fast_optical_flow(g_src, g_dst):
    """
    Tính Farneback Optical Flow siêu tốc ở độ phân giải 1/2 (480x270).
    Tính luôn magnitude và trả về (flow, mag) để tái sử dụng trong cache mà không phải gọi cv2.cartToPolar.
    """
    h, w = g_src.shape[:2]
    w_half, h_half = w // 2, h // 2
    g_s_small = cv2.resize(g_src, (w_half, h_half), interpolation=cv2.INTER_AREA)
    g_d_small = cv2.resize(g_dst, (w_half, h_half), interpolation=cv2.INTER_AREA)
    
    flow_small = cv2.calcOpticalFlowFarneback(
        g_s_small, g_d_small, None,
        pyr_scale=0.5, levels=2, winsize=11, iterations=2, poly_n=5, poly_sigma=1.1, flags=0
    )
    flow = cv2.resize(flow_small * 2.0, (w, h), interpolation=cv2.INTER_LINEAR)
    mag = cv2.magnitude(flow[..., 0], flow[..., 1])
    return flow, mag

def calc_sspr_fast(m0, mgt, m1):
    """
    Kiểm tra nhanh Static Stroke Pixel Ratio (sspr) từ 3 stroke masks (0.2 ms).
    """
    active_mask = (m0 | mgt | m1)
    tot_active = np.sum(active_mask)
    if tot_active == 0:
        return 0.0
    static_overlap = np.sum(m0 & mgt & m1)
    return float(static_overlap) / float(tot_active)

_REMAP_GRID_CACHE = {}

def get_remap_grid(h, w):
    key = (h, w)
    if key not in _REMAP_GRID_CACHE:
        gx, gy = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
        _REMAP_GRID_CACHE[key] = (gx, gy)
    return _REMAP_GRID_CACHE[key]

def calc_motion_kinematics_fast(g0, ggt, g1, line0, linegt, line1, flow_0_gt=None, flow_gt_1=None):
    """
    Tính động học chuyển động siêu tốc hỗ trợ tái sử dụng Optical Flow từ Cache và Early-Exit 2 tầng.
    - Nhận (flow, mag) từ cache hoặc tự tính.
    - Early-Exit 1: Nếu disp_0_gt < M1_DSMD_MIN, bỏ qua luôn chặng 2 (flow_gt_1).
    - Early-Exit 2: Nếu không đạt dsmd, mbr, mdcc, bỏ qua luôn cv2.remap.
    - cv2.remap sử dụng grid float32 được cache sẵn theo (h, w).
    """
    m0 = (line0 < DARK_THRESH) if line0.dtype != bool else line0
    mgt = (linegt < DARK_THRESH) if linegt.dtype != bool else linegt
    m1 = (line1 < DARK_THRESH) if line1.dtype != bool else line1

    if np.sum(m0) == 0 or np.sum(mgt) == 0 or np.sum(m1) == 0:
        return {'dsmd': 0.0, 'mbr': 0.0, 'sspr': 1.0, 'mdcc': 0.0, 'wsds': 0.0, 'valid': False}, flow_0_gt, flow_gt_1

    # 1. Chặng 1: (0 -> gt)
    if flow_0_gt is None:
        flow_0_gt, mag_0_gt = calc_fast_optical_flow(g0, ggt)
    else:
        if isinstance(flow_0_gt, tuple):
            flow_0_gt, mag_0_gt = flow_0_gt
        else:
            mag_0_gt = cv2.magnitude(flow_0_gt[..., 0], flow_0_gt[..., 1])

    disp_0_gt = float(np.mean(mag_0_gt[m0]))

    # Early-Exit 1: Nếu chặng 1 đứng yên (< M1_DSMD_MIN), thoát ngay mà KHÔNG cần tính flow_gt_1!
    if disp_0_gt < THRESHOLDS['M1_DSMD_MIN']:
        return {'dsmd': disp_0_gt, 'mbr': 0.0, 'sspr': 0.0, 'mdcc': 0.0, 'wsds': 0.0, 'valid': False}, (flow_0_gt, mag_0_gt), flow_gt_1

    # 2. Chặng 2: (gt -> 1)
    if flow_gt_1 is None:
        flow_gt_1, mag_gt_1 = calc_fast_optical_flow(ggt, g1)
    else:
        if isinstance(flow_gt_1, tuple):
            flow_gt_1, mag_gt_1 = flow_gt_1
        else:
            mag_gt_1 = cv2.magnitude(flow_gt_1[..., 0], flow_gt_1[..., 1])

    disp_gt_1 = float(np.mean(mag_gt_1[mgt]))
    dsmd = min(disp_0_gt, disp_gt_1)
    mbr = dsmd / max(max(disp_0_gt, disp_gt_1), 1e-6)

    dot_p = flow_0_gt[..., 0] * flow_gt_1[..., 0] + flow_0_gt[..., 1] * flow_gt_1[..., 1]
    denom = mag_0_gt * mag_gt_1 + 1e-6
    cos_field = dot_p / denom
    mdcc = float(np.mean(cos_field[mgt]))

    # Early-Exit 2: Kiểm tra các ngưỡng động học trước khi chạy cv2.remap
    if (dsmd < THRESHOLDS['M1_DSMD_MIN'] or dsmd > THRESHOLDS['M1_DSMD_MAX'] or
        mbr < THRESHOLDS['M2_MBR_MIN'] or mdcc < THRESHOLDS['M4_MDCC_MIN']):
        return {'dsmd': dsmd, 'mbr': mbr, 'sspr': 0.0, 'mdcc': mdcc, 'wsds': 0.0, 'valid': False}, (flow_0_gt, mag_0_gt), (flow_gt_1, mag_gt_1)

    # 3. Warped Stroke Dice Score (wsds) với lưới tọa độ đã cache sẵn
    h, w = g0.shape[:2]
    grid_x, grid_y = get_remap_grid(h, w)
    map_x = grid_x - flow_0_gt[..., 0]
    map_y = grid_y - flow_0_gt[..., 1]
    warped_m0 = cv2.remap(m0.astype(np.uint8), map_x, map_y, interpolation=cv2.INTER_NEAREST)

    intersection = np.sum(warped_m0 & mgt)
    sum_strokes = np.sum(warped_m0) + np.sum(mgt)
    wsds = float(2.0 * intersection) / float(max(sum_strokes, 1))

    return {'dsmd': dsmd, 'mbr': mbr, 'sspr': 0.0, 'mdcc': mdcc, 'wsds': wsds, 'valid': True}, (flow_0_gt, mag_0_gt), (flow_gt_1, mag_gt_1)

print("[✓] Quality Metrics Engine đã sẵn sàng với Fast Optical Flow Engine!")


---
### 📦 Bước 5: Định Vị Thông Minh Dataset Candidate ATD-12K & Tự Động Kế Thừa Chỉ Số Clip (`clip_004166+`)
> **CƠ CHẾ KẾ THỪA CHỈ SỐ CLIP AN TOÀN TUYỆT ĐỐI:**
> - Part 1 & Part 2 trước đó đã tạo tổng cộng **4,165 Golden Triplets** (`clip_000001` đến `clip_004165`).
> - Cell này sẽ tự động kiểm tra xem bạn có tải kèm file `triplets_combined.csv` cũ vào `/kaggle/input` hay không:
>   - Nếu có: Tự động trích xuất chỉ số lớn nhất và cộng tiếp $1$ (ví dụ: $4165 + 1 = 4166$).
>   - Nếu không tải kèm: Mặc định an toàn bắt đầu từ **`clip_004166`**.
> - Nhờ vậy, Part 3 xuất ra sẽ có định danh liên tục từ **`clip_004166` $\to$ `clip_XXXXXX`**, hoàn toàn tương thích để nối tiếp vào Golden Dataset!


In [ ]:
WORKING_DIR = "/kaggle/working"
FINAL_ZIP_PART3 = os.path.join(WORKING_DIR, "sakuga_golden_dataset_tps_part3.zip")
CSV_PART3_PATH = os.path.join(WORKING_DIR, "triplets_part3.csv")
AUDIT_PART3_PATH = os.path.join(WORKING_DIR, "quality_audit_report_part3.csv")
COMBINED_ALL_CSV = os.path.join(WORKING_DIR, "triplets_combined_all.csv")

# Mặc định chỉ số bắt đầu cho Part 3 (nối tiếp sau 4,165 clip của Part 1 & Part 2)
DEFAULT_START_CLIP_IDX = 4166

# Tự động quét tìm file triplets_combined.csv của Part 1 & 2 nếu có trong /kaggle/input
existing_combined_csv = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f in ["triplets_combined.csv", "triplets_master.csv"]:
            existing_combined_csv = os.path.join(root, f)
            break
    if existing_combined_csv:
        break

start_p3_idx = DEFAULT_START_CLIP_IDX
prev_triplets_data = []

if existing_combined_csv and os.path.exists(existing_combined_csv):
    try:
        with open(existing_combined_csv, 'r', encoding='utf-8') as f:
            prev_triplets_data = list(csv.DictReader(f))
        max_idx = 0
        for row in prev_triplets_data:
            cid = row.get('clip_id', '')
            if cid.startswith('clip_'):
                try:
                    num = int(cid.split('_')[1])
                    if num > max_idx:
                        max_idx = num
                except Exception:
                    pass
        if max_idx > 0:
            start_p3_idx = max_idx + 1
            print(f"[✓] Đã tìm thấy master CSV trước đó ({len(prev_triplets_data):,} clips, max: clip_{max_idx:06d}).")
    except Exception as e:
        print(f"[!] Lỗi đọc {existing_combined_csv}: {e}")

print(f"👉 CHỈ SỐ BẮT ĐẦU CHO PART 3: clip_{start_p3_idx:06d}")

# Lớp đọc Dataset Candidate ATD-12K
class CandidateDatasetSource:
    def __init__(self, search_root="/kaggle/input"):
        self.folder_root = None
        self.manifest_data = []
        self._discover(search_root)

    def _discover(self, search_root):
        candidate_dirs = []
        # Quét tìm tất cả thư mục có chứa dataset_manifest.json hoặc atd12k_manifest.json
        for root, dirs, files in os.walk(search_root):
            m_files = [f for f in files if f.endswith(".json") and "manifest" in f.lower()]
            if m_files:
                candidate_dirs.append((root, m_files[0]))

        # Thử nạp từng thư mục
        for p_dir, mf_name in candidate_dirs:
            mf_p = os.path.join(p_dir, mf_name)
            try:
                with open(mf_p, "r", encoding="utf-8") as f:
                    data = json.load(f)
                if isinstance(data, list) and len(data) > 0:
                    sample = data[0]
                    if "frame1" in sample and ("video_name" in sample or "triplet_id" in sample):
                        self.folder_root = p_dir
                        self.manifest_data = data
                        print(f"[✓] Đã nhận diện Candidate Dataset tại: '{self.folder_root}'")
                        print(f"    + File Manifest : {mf_name}")
                        print(f"    + Tổng số mẫu   : {len(data):,} candidate triplets")
                        return
            except Exception:
                continue

        if not self.folder_root:
            print(f"[!] CẢNH BÁO: Chưa tìm thấy dataset candidate nào trong {search_root}!")
            print("    -> Hãy chắc chắn bạn đã nhấn 'Add Input' và chọn dataset chứa atd12k_candidates_540p.zip!")

    def exists(self, rel_path):
        if not self.folder_root:
            return False
        p = os.path.join(self.folder_root, rel_path)
        if os.path.exists(p):
            return True
        p_alt = os.path.join(self.folder_root, os.path.basename(os.path.dirname(rel_path)), os.path.basename(rel_path))
        return os.path.exists(p_alt)

    def read_image(self, rel_path):
        if not self.folder_root:
            return None
        p = os.path.join(self.folder_root, rel_path)
        if not os.path.exists(p):
            p = os.path.join(self.folder_root, os.path.basename(os.path.dirname(rel_path)), os.path.basename(rel_path))
        if not os.path.exists(p):
            return None
        return cv2.imread(p)

source_part3 = CandidateDatasetSource()

# Xác thực các candidate tồn tại trên đĩa
valid_candidates_p3 = []
if source_part3.manifest_data:
    print("[*] Đang xác thực các Candidate Triplets tồn tại trên đĩa...")
    for tr in tqdm(source_part3.manifest_data, desc="Verifying Candidates", file=sys.stdout):
        if source_part3.exists(tr["frame1"]):
            valid_candidates_p3.append(tr)
    print(f"[✓] Sẵn sàng thực thi {len(valid_candidates_p3):,} candidate triplets cho Part 3!")


---
### 🚀 Bước 6: Động Cơ Thực Thi Song Song 2x GPU (`run_golden_batch_to_zip`)


In [ ]:
SCREEN_MIN_INLIERS = 20
SCREEN_MAX_MEAN_DISP = 15.0

def run_golden_batch_to_zip(
    dataset_source,
    batch_candidates,
    part_label,
    final_zip_path,
    gpu_workers,
    start_clip_idx=DEFAULT_START_CLIP_IDX
):
    part_slug = part_label.lower().replace(" ", "")
    staging_dir = os.path.join(WORKING_DIR, f"_staging_{part_slug}")
    staging_clips = os.path.join(staging_dir, "clips")
    staging_matches = os.path.join(staging_dir, "matches")

    os.makedirs(staging_clips, exist_ok=True)
    os.makedirs(staging_matches, exist_ok=True)

    # 1. Gom nhóm candidates theo video_name để tận dụng Frame-Pool Cache
    video_groups = defaultdict(list)
    for c in batch_candidates:
        v_key = c.get('video_name') or c.get('source') or c.get('triplet_id', 'clip_unknown')
        video_groups[v_key].append(c)

    video_group_list = list(video_groups.items())
    video_group_list.sort(key=lambda x: len(x[1]), reverse=True)

    chunks = [[] for _ in range(len(gpu_workers))]
    chunk_loads = [0] * len(gpu_workers)
    for v_name, v_triplets in video_group_list:
        min_idx = chunk_loads.index(min(chunk_loads))
        chunks[min_idx].append((v_name, v_triplets))
        chunk_loads[min_idx] += len(v_triplets)

    print(f"\n" + "=" * 80)
    print(f"[*] BẮT ĐẦU XỬ LÝ {part_label.upper()} TRÊN 2x GPU TESLA T4...")
    print(f"    + Tổng số nhóm video/scene: {len(video_groups):,} nhóm")
    print(f"    + Tổng Candidate Triplets : {len(batch_candidates):,} triplets")
    for i, chk in enumerate(chunks):
        n_vids = len(chk)
        n_trips = chunk_loads[i]
        print(f"    + Worker {i} ({gpu_workers[i].device}): Nhận {n_vids:,} scenes (~{n_trips:,} triplets)")
    print(f"    + Chỉ số bắt đầu (Start)  : clip_{start_clip_idx:06d}")
    print("=" * 80)

    start_time = time.time()

    def process_chunk_worker(worker_id, chunk_videos, worker):
        try:
            if 'cuda' in worker.device.type:
                torch.cuda.set_device(worker.device)
            cv2.setNumThreads(1)

            local_golden = []
            total_triplets_chunk = sum(len(tr_list) for _, tr_list in chunk_videos)
            pbar = tqdm(total=total_triplets_chunk, desc=f"{part_label} - GPU {worker_id} ({worker.device})", position=worker_id, file=sys.stdout)

            for v_idx, (v_name, triplets_of_video) in enumerate(chunk_videos):
                try:
                    needed_paths = set()
                    for tr in triplets_of_video:
                        needed_paths.add(tr["frame1"])
                        needed_paths.add(tr["frame2"])
                        needed_paths.add(tr["frame3"])
                    unique_paths = sorted(list(needed_paths))

                    raw_imgs_dict = {}
                    def _read_img(p):
                        return p, dataset_source.read_image(p)

                    if len(unique_paths) > 0:
                        with ThreadPoolExecutor(max_workers=min(4, len(unique_paths))) as read_pool:
                            for p, img in read_pool.map(_read_img, unique_paths):
                                if img is not None and isinstance(img, np.ndarray) and img.size > 0:
                                    raw_imgs_dict[p] = img

                    valid_paths = [p for p in unique_paths if p in raw_imgs_dict]
                    raw_imgs_list = [raw_imgs_dict[p] for p in valid_paths]

                    frame_cache = {}
                    if len(raw_imgs_list) > 0:
                        if hasattr(worker, 'bg_remover'):
                            bg_results = worker.bg_remover.remove_background_batch(raw_imgs_list, batch_size=16)
                        else:
                            bg_results = worker.remove_background_batch(raw_imgs_list, batch_size=16)

                        passed_bg_indices = []
                        for f_i, p in enumerate(valid_paths):
                            white_img, fg_mask, fg_ratio = bg_results[f_i]
                            if 0.03 <= fg_ratio <= 0.85:
                                passed_bg_indices.append(f_i)
                            else:
                                frame_cache[p] = {'valid': False}

                        if len(passed_bg_indices) > 0:
                            white_passed = [bg_results[idx][0] for idx in passed_bg_indices]
                            line_results = worker.extract_lineart_and_edt_batch(white_passed, batch_size=16)

                            for sub_idx, f_i in enumerate(passed_bg_indices):
                                p = valid_paths[f_i]
                                white_img, fg_mask, fg_ratio = bg_results[f_i]
                                line_img, swdr, cp = line_results[sub_idx]

                                if not (
                                    (cp['pwcr'] >= THRESHOLDS['B1_PWCR_MIN']) and
                                    (cp['rgd'] <= THRESHOLDS['B2_RGD_MAX']) and
                                    (swdr <= THRESHOLDS['V1_SWDR_MAX'])
                                ):
                                    frame_cache[p] = {'valid': False}
                                    continue

                                topo = calc_stroke_topology(line_img)
                                frame_valid = (
                                    (topo['dccr'] >= THRESHOLDS['C1_DCCR_MIN']) and
                                    (topo['mccc'] <= THRESHOLDS['C2_MCCC_MAX']) and
                                    (topo['tnfar'] <= THRESHOLDS['C3_TNFAR_MAX'])
                                )
                                if not frame_valid:
                                    frame_cache[p] = {'valid': False}
                                    continue

                                white_gray = cv2.cvtColor(white_img, cv2.COLOR_BGR2GRAY)
                                stroke_mask = (line_img < DARK_THRESH)

                                frame_cache[p] = {
                                    'valid': True,
                                    'white': white_img,
                                    'white_gray': white_gray,
                                    'fg_mask': fg_mask,
                                    'fg_ratio': fg_ratio,
                                    'line': line_img,
                                    'stroke_mask': stroke_mask,
                                    'cp': cp,
                                    'topo': topo,
                                    'swdr': swdr,
                                    'gs_feats': None
                                }

                            del white_passed, line_results
                        del bg_results
                    del raw_imgs_dict, raw_imgs_list

                    gs_pair_cache = {}
                    flow_pair_cache = {}

                    for tr in triplets_of_video:
                        p0_path = tr["frame1"]
                        pgt_path = tr["frame2"]
                        p1_path = tr["frame3"]

                        f0_info = frame_cache.get(p0_path)
                        fgt_info = frame_cache.get(pgt_path)
                        f1_info = frame_cache.get(p1_path)

                        if (f0_info is None or fgt_info is None or f1_info is None or
                            not (f0_info['valid'] and fgt_info['valid'] and f1_info['valid'])):
                            pbar.update(1)
                            continue

                        bspi, _, _ = calc_bspi(f0_info['cp']['density'], fgt_info['cp']['density'], f1_info['cp']['density'])
                        if bspi > THRESHOLDS['B3_BSPI_MAX']:
                            pbar.update(1)
                            continue

                        sspr = calc_sspr_fast(f0_info['stroke_mask'], fgt_info['stroke_mask'], f1_info['stroke_mask'])
                        if sspr > THRESHOLDS['M3_SSPR_MAX']:
                            pbar.update(1)
                            continue

                        # GATE 0: GlueStick Matching
                        pair_key = (p0_path, p1_path)
                        if pair_key in gs_pair_cache:
                            gs_match_res = gs_pair_cache[pair_key]
                        else:
                            if f0_info['gs_feats'] is None:
                                f0_info['gs_feats'] = worker.extract_gluestick_feats(f0_info['white_gray'])
                            if f1_info['gs_feats'] is None:
                                f1_info['gs_feats'] = worker.extract_gluestick_feats(f1_info['white_gray'])

                            g0 = f0_info['white_gray']
                            h, w = g0.shape[:2]
                            fg_mask0 = f0_info['fg_mask']
                            fg_mask1 = f1_info['fg_mask']

                            pred_gs = worker.match_gluestick_pair(f0_info['gs_feats'], f1_info['gs_feats'], (h, w))

                            m_kpts0 = pred_gs['matches0'][0].cpu().numpy()
                            kpts0 = pred_gs['keypoints0'][0].cpu().numpy()
                            kpts1 = pred_gs['keypoints1'][0].cpu().numpy()

                            valid_m = m_kpts0 > -1
                            p0 = kpts0[valid_m]
                            p1 = kpts1[m_kpts0[valid_m]]

                            if len(p0) < SCREEN_MIN_INLIERS:
                                gs_match_res = {'valid': False}
                            else:
                                x0_idx = np.clip(np.round(p0[:, 0]).astype(int), 0, w - 1)
                                y0_idx = np.clip(np.round(p0[:, 1]).astype(int), 0, h - 1)
                                x1_idx = np.clip(np.round(p1[:, 0]).astype(int), 0, w - 1)
                                y1_idx = np.clip(np.round(p1[:, 1]).astype(int), 0, h - 1)

                                fg_keep = fg_mask0[y0_idx, x0_idx] & fg_mask1[y1_idx, x1_idx]
                                p0 = p0[fg_keep]
                                p1 = p1[fg_keep]

                                if len(p0) < SCREEN_MIN_INLIERS:
                                    gs_match_res = {'valid': False}
                                else:
                                    M, inliers = cv2.estimateAffinePartial2D(
                                        p0.astype(np.float32), 
                                        p1.astype(np.float32), 
                                        method=cv2.RANSAC, 
                                        ransacReprojThreshold=5.0,
                                        maxIters=500,
                                        confidence=0.98
                                    )
                                    if inliers is None:
                                        gs_match_res = {'valid': False}
                                    else:
                                        mask_inl = inliers.flatten().astype(bool)
                                        inliers0 = p0[mask_inl]
                                        inliers1 = p1[mask_inl]

                                        if len(inliers0) < SCREEN_MIN_INLIERS:
                                            gs_match_res = {'valid': False}
                                        else:
                                            disps = np.linalg.norm(inliers1 - inliers0, axis=1)
                                            mean_disp = float(np.mean(disps))
                                            max_disp = float(np.max(disps))

                                            if mean_disp > SCREEN_MAX_MEAN_DISP or max_disp > 40.0:
                                                gs_match_res = {'valid': False}
                                            else:
                                                flow_norm0 = inliers0.copy()
                                                flow_norm0[:, 0] /= float(w)
                                                flow_norm0[:, 1] /= float(h)
                                                flow_norm1 = inliers1.copy()
                                                flow_norm1[:, 0] /= float(w)
                                                flow_norm1[:, 1] /= float(h)
                                                kp_matches = np.stack([flow_norm0, flow_norm1], axis=0)

                                                gs_match_res = {
                                                    'valid': True,
                                                    'inliers0': inliers0,
                                                    'inliers1': inliers1,
                                                    'mean_disp': mean_disp,
                                                    'kp_matches': kp_matches
                                                }

                            gs_pair_cache[pair_key] = gs_match_res

                        if not gs_match_res['valid']:
                            pbar.update(1)
                            continue

                        inliers0 = gs_match_res['inliers0']
                        inliers1 = gs_match_res['inliers1']
                        mean_disp = gs_match_res['mean_disp']
                        kp_matches = gs_match_res['kp_matches']
                        fg_maskgt = fgt_info['fg_mask']
                        h, w = fg_maskgt.shape[:2]

                        t_ratio = float(tr.get('t', 0.5))
                        p_mid = (1.0 - t_ratio) * inliers0 + t_ratio * inliers1
                        x_mid = np.clip(np.round(p_mid[:, 0]).astype(int), 0, w - 1)
                        y_mid = np.clip(np.round(p_mid[:, 1]).astype(int), 0, h - 1)
                        in_gt = fg_maskgt[y_mid, x_mid]
                        if np.mean(in_gt) < 0.60:
                            pbar.update(1)
                            continue

                        # GATE 3: Fast Optical Flow & Warped Dice
                        flow_0_key = (p0_path, pgt_path)
                        flow_1_key = (pgt_path, p1_path)
                        cached_f0 = flow_pair_cache.get(flow_0_key)
                        cached_f1 = flow_pair_cache.get(flow_1_key)

                        mot, f0_computed, f1_computed = calc_motion_kinematics_fast(
                            f0_info['white_gray'], fgt_info['white_gray'], f1_info['white_gray'],
                            f0_info['stroke_mask'], fgt_info['stroke_mask'], f1_info['stroke_mask'],
                            flow_0_gt=cached_f0, flow_gt_1=cached_f1
                        )
                        if cached_f0 is None and f0_computed is not None:
                            flow_pair_cache[flow_0_key] = f0_computed
                        if cached_f1 is None and f1_computed is not None:
                            flow_pair_cache[flow_1_key] = f1_computed

                        if (not mot.get('valid', True) or
                            mot['dsmd'] < THRESHOLDS['M1_DSMD_MIN'] or mot['dsmd'] > THRESHOLDS['M1_DSMD_MAX'] or
                            mot['mbr'] < THRESHOLDS['M2_MBR_MIN'] or
                            mot['mdcc'] < THRESHOLDS['M4_MDCC_MIN'] or mot['wsds'] < THRESHOLDS['C4_WSDS_MIN']):
                            pbar.update(1)
                            continue

                        # ĐẠT CHUẨN GOLDEN TRIPLET PART 3
                        tid = tr.get("triplet_id", f"tr_{len(local_golden)+1:06d}")
                        unique_tmp_id = f"tmp_w{worker_id}_{len(local_golden)+1:05d}"
                        c_dir = os.path.join(staging_clips, unique_tmp_id)
                        os.makedirs(c_dir, exist_ok=True)

                        png_params = [cv2.IMWRITE_PNG_COMPRESSION, 1]
                        cv2.imwrite(os.path.join(c_dir, "000001.png"), f0_info['line'], png_params)
                        cv2.imwrite(os.path.join(c_dir, "000002.png"), fgt_info['line'], png_params)
                        cv2.imwrite(os.path.join(c_dir, "000003.png"), f1_info['line'], png_params)

                        m_path = os.path.join(staging_matches, f"match_{unique_tmp_id}.npy")
                        np.save(m_path, kp_matches)

                        pwcr_min = min(f0_info['cp']['pwcr'], fgt_info['cp']['pwcr'], f1_info['cp']['pwcr'])
                        dccr_min = min(f0_info['topo']['dccr'], fgt_info['topo']['dccr'], f1_info['topo']['dccr'])
                        mccc_max = max(f0_info['topo']['mccc'], fgt_info['topo']['mccc'], f1_info['topo']['mccc'])

                        local_golden.append({
                            'temp_id': unique_tmp_id,
                            'orig_triplet_id': tid,
                            'video_name': tr.get('video_name') or tr.get('source', ''),
                            'delta_f': tr.get('delta_frames', tr.get('delta_f', 2)),
                            't': tr.get('t', 0.5),
                            'inliers_count': int(len(inliers0)),
                            'mean_disp': round(mean_disp, 2),
                            'pwcr_min': round(pwcr_min, 4),
                            'dccr_min': round(dccr_min, 4),
                            'wsds': round(mot['wsds'], 4),
                            'dsmd': round(mot['dsmd'], 2),
                            'mbr': round(mot['mbr'], 2),
                            'bspi': round(bspi, 4),
                            'mccc': int(mccc_max),
                            'part': part_label
                        })

                        pbar.update(1)

                    frame_cache.clear()
                    gs_pair_cache.clear()
                    flow_pair_cache.clear()
                    del frame_cache, gs_pair_cache, flow_pair_cache

                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

                    pbar.set_postfix({'Golden': len(local_golden)})

                except Exception as e_vid:
                    print(f"\n[!] Cảnh báo GPU {worker_id}: Bỏ qua scene '{v_name}' do: {e_vid}", file=sys.stdout)
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    pbar.update(len(triplets_of_video))
                    continue

            pbar.close()
            return local_golden
        except Exception as e_worker:
            import traceback
            print(f"\n[CRITICAL ERROR] Worker {worker_id} trên {worker.device} gặp sự cố: {e_worker}\n{traceback.format_exc()}", file=sys.stdout)
            return local_golden

    with ThreadPoolExecutor(max_workers=len(gpu_workers)) as executor:
        futures = {
            executor.submit(process_chunk_worker, i, chunks[i], gpu_workers[i]): i
            for i in range(len(gpu_workers))
        }
        all_golden_raw = []
        from concurrent.futures import as_completed
        for f in as_completed(futures):
            w_id = futures[f]
            try:
                res = f.result()
                all_golden_raw.extend(res)
                print(f"\n[✓] Worker {w_id} ({gpu_workers[w_id].device}) hoàn thành xuất sắc ({len(res):,} golden triplets)!", file=sys.stdout)
            except Exception as e_f:
                import traceback
                print(f"\n[!] Worker {w_id} lỗi khi thu thập kết quả: {e_f}\n{traceback.format_exc()}", file=sys.stdout)

    print(f"\n[*] Thu hoạch thành công {len(all_golden_raw):,} Golden Triplets từ {part_label}!")

    if len(all_golden_raw) == 0:
        print("[!] Không có Golden Triplet nào đạt chuẩn. Vui lòng kiểm tra lại dataset đầu vào!")
        shutil.rmtree(staging_dir, ignore_errors=True)
        return [], []

    # 2. Đánh số Canonical Clip ID liên tục (bắt đầu từ start_clip_idx)
    all_golden_raw.sort(key=lambda x: (x.get('video_name', ''), x['t']))
    for idx, g in enumerate(all_golden_raw):
        g['canonical_clip_id'] = f"clip_{start_clip_idx + idx:06d}"

    # 3. Phân chia Train (85%) / Val (15%) Stratified Split với seed 42
    random.seed(42)
    groups_by_t = {}
    for g in all_golden_raw:
        t_key = round(float(g['t']), 2)
        groups_by_t.setdefault(t_key, []).append(g)

    train_items = []
    val_items = []
    for t_key, items in groups_by_t.items():
        random.shuffle(items)
        n_val = max(1, int(len(items) * 0.15)) if len(items) > 1 else 0
        val_items.extend(items[:n_val])
        train_items.extend(items[n_val:])

    for item in train_items:
        item['split'] = 'train'
    for item in val_items:
        item['split'] = 'val'

    all_packaged_meta = train_items + val_items
    all_packaged_meta.sort(key=lambda x: x['canonical_clip_id'])

    # 4. Tạo metadata CSV
    triplets_csv_rows = []
    audit_rows = []
    for item in all_packaged_meta:
        cid = item['canonical_clip_id']
        s = item['split']
        triplets_csv_rows.append({
            'split': s,
            'clip_id': cid,
            't': item['t'],
            'delta_f': item['delta_f'],
            'video_name': item['video_name'],
            'frame0': f"{s}/{cid}/000001.png",
            'frame_gt': f"{s}/{cid}/000002.png",
            'frame1': f"{s}/{cid}/000003.png",
            'match_file': f"matches/{cid}/match_000001_000003.npy",
            'mean_disp': item['mean_disp'],
            'inliers': item['inliers_count'],
            'wsds': item['wsds'],
            'pwcr': item['pwcr_min'],
            'dccr': item['dccr_min']
        })
        audit_rows.append(item)

    # 5. Đóng gói trực tiếp vào file ZIP thành phẩm
    if os.path.exists(final_zip_path):
        os.remove(final_zip_path)

    print(f"[*] Đang đóng gói nén trực tiếp vào: {final_zip_path}...")
    with zipfile.ZipFile(final_zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
        for item in tqdm(all_packaged_meta, desc=f"Zipping {part_label}", file=sys.stdout):
            cid = item['canonical_clip_id']
            tmp_id = item['temp_id']
            s = item['split']

            src_clip_dir = os.path.join(staging_clips, tmp_id)
            for fname in ["000001.png", "000002.png", "000003.png"]:
                fpath = os.path.join(src_clip_dir, fname)
                if os.path.exists(fpath):
                    zipf.write(fpath, arcname=f"{s}/{cid}/{fname}")

            src_m = os.path.join(staging_matches, f"match_{tmp_id}.npy")
            if os.path.exists(src_m):
                zipf.write(src_m, arcname=f"matches/{cid}/match_000001_000003.npy")

        if triplets_csv_rows:
            import io
            buf_csv = io.StringIO()
            writer = csv.DictWriter(buf_csv, fieldnames=list(triplets_csv_rows[0].keys()))
            writer.writeheader()
            writer.writerows(triplets_csv_rows)
            zipf.writestr("triplets.csv", buf_csv.getvalue().encode('utf-8'))
            zipf.writestr(f"triplets_{part_slug}.csv", buf_csv.getvalue().encode('utf-8'))

    shutil.rmtree(staging_dir, ignore_errors=True)

    elapsed_min = (time.time() - start_time) / 60.0
    zip_size_mb = os.path.getsize(final_zip_path) / (1024**2)
    tot_d, usd_d, free_d = shutil.disk_usage(WORKING_DIR)

    print("\n" + "=" * 80)
    print(f"🎉 HOÀN TẤT THÀNH CÔNG {part_label.upper()}!")
    print(f"   📁 File thành phẩm       : {final_zip_path}")
    print(f"   📦 Kích thước file ZIP   : {zip_size_mb:.2f} MB")
    print(f"   📊 Số Golden Triplets    : {len(all_packaged_meta):,} clips (Train: {len(train_items):,} | Val: {len(val_items):,})")
    print(f"   ⏱️ Thời gian thực thi    : {elapsed_min:.2f} phút")
    print(f"   💾 Dung lượng đĩa trống  : {free_d/(1024**3):.2f} GB / {tot_d/(1024**3):.2f} GB")
    print("=" * 80)

    return all_packaged_meta, audit_rows


---
### 🔍 Bước 7: Tiện Ích Sanity Check Trực Quan Từ File ZIP Thành Phẩm (`inspect_golden_zip_sample`)


In [ ]:
def inspect_golden_zip_sample(zip_path, part_name):
    if not os.path.exists(zip_path):
        print(f"[!] Không tìm thấy file {zip_path}")
        return

    with zipfile.ZipFile(zip_path, 'r') as z:
        file_list = z.namelist()
        sample_f0_candidates = [f for f in file_list if f.endswith("000001.png") and "train/" in f]
        if not sample_f0_candidates:
            sample_f0_candidates = [f for f in file_list if f.endswith("000001.png")]
        if not sample_f0_candidates:
            print(f"[!] {part_name}: Chưa tìm thấy mẫu clip nào trong file ZIP!")
            return

        sample_f0 = sample_f0_candidates[min(5, len(sample_f0_candidates) - 1)]
        sample_fgt = sample_f0.replace("000001.png", "000002.png")
        sample_f1 = sample_f0.replace("000001.png", "000003.png")

        f0_bytes = z.read(sample_f0)
        fgt_bytes = z.read(sample_fgt)
        f1_bytes = z.read(sample_f1)

        img0 = cv2.imdecode(np.frombuffer(f0_bytes, np.uint8), cv2.IMREAD_GRAYSCALE)
        imggt = cv2.imdecode(np.frombuffer(fgt_bytes, np.uint8), cv2.IMREAD_GRAYSCALE)
        img1 = cv2.imdecode(np.frombuffer(f1_bytes, np.uint8), cv2.IMREAD_GRAYSCALE)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].imshow(img0, cmap='gray')
        axes[0].set_title(f"Frame 0 (Hairline Smooth Lineart)\n{sample_f0}", fontsize=11)
        axes[0].axis("off")

        axes[1].imshow(imggt, cmap='gray')
        axes[1].set_title(f"Frame GT (Ground Truth Lineart)\n{sample_fgt}", fontsize=11, color='green', fontweight='bold')
        axes[1].axis("off")

        axes[2].imshow(img1, cmap='gray')
        axes[2].set_title(f"Frame 1 (Hairline Smooth Lineart)\n{sample_f1}", fontsize=11)
        axes[2].axis("off")

        clip_name = os.path.basename(os.path.dirname(sample_f0))
        plt.suptitle(f"MẪU NGHIỆM THU CHUẨN VÀNG ({part_name}): {clip_name}", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        print(f"[✓] Sanity check {part_name} hoàn tất: Dữ liệu đạt độ sắc nét và tinh khiết chuẩn vàng 100%!")


---
### 🎯 Bước 8: [THỰC THI PART 3] Chạy Toàn Bộ Candidate ATD-12K -> Xuất File `sakuga_golden_dataset_tps_part3.zip`
> **LƯU Ý:**
> 1. Nhấn chạy code cell này để xử lý toàn bộ Candidate Triplets của **Part 3**.
> 2. Các clip mới sẽ được đánh số nối tiếp bắt đầu từ **`clip_004166`** (hoặc tiếp nối chỉ số lớn nhất nếu bạn có đính kèm file CSV cũ).
> 3. Sau khi hoàn tất: Tải các file thành phẩm từ panel Output bên phải về máy tính:
>    - `sakuga_golden_dataset_tps_part3.zip`
>    - `triplets_part3.csv`
>    - `quality_audit_report_part3.csv`


In [ ]:
# ==============================================================================
# THỰC THI BATCH PART 3 TRÊN 2x GPU TESLA T4
# ==============================================================================
meta_part3, audit_part3 = run_golden_batch_to_zip(
    source_part3,
    valid_candidates_p3,
    "Part 3",
    FINAL_ZIP_PART3,
    gpu_workers,
    start_clip_idx=start_p3_idx
)

# 1. Lưu metadata JSON Part 3
META_P3_PATH = os.path.join(WORKING_DIR, "golden_meta_part3.json")
with open(META_P3_PATH, "w", encoding="utf-8") as f:
    json.dump(meta_part3, f, indent=2, ensure_ascii=False)

# 2. Xuất file triplets_part3.csv
if meta_part3:
    csv_rows_p3 = []
    for item in meta_part3:
        cid = item['canonical_clip_id']
        s = item['split']
        csv_rows_p3.append({
            'split': s,
            'clip_id': cid,
            't': item['t'],
            'delta_f': item['delta_f'],
            'video_name': item['video_name'],
            'frame0': f"{s}/{cid}/000001.png",
            'frame_gt': f"{s}/{cid}/000002.png",
            'frame1': f"{s}/{cid}/000003.png",
            'match_file': f"matches/{cid}/match_000001_000003.npy",
            'mean_disp': item['mean_disp'],
            'inliers': item['inliers_count'],
            'wsds': item['wsds'],
            'pwcr': item['pwcr_min'],
            'dccr': item['dccr_min']
        })

    with open(CSV_PART3_PATH, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(csv_rows_p3[0].keys()))
        writer.writeheader()
        writer.writerows(csv_rows_p3)

    with open(AUDIT_PART3_PATH, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(meta_part3[0].keys()))
        writer.writeheader()
        writer.writerows(meta_part3)

# 3. Gộp toàn bộ với Part 1 & 2 nếu có file CSV trước đó
if prev_triplets_data and meta_part3:
    all_combined_rows = list(prev_triplets_data) + csv_rows_p3
    with open(COMBINED_ALL_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(all_combined_rows[0].keys()))
        writer.writeheader()
        writer.writerows(all_combined_rows)
    print(f"[✓] Đã tạo file master CSV gộp toàn diện cả 3 phần ({len(all_combined_rows):,} clips): {COMBINED_ALL_CSV}")

p3_size_mb = os.path.getsize(FINAL_ZIP_PART3) / (1024**2) if os.path.exists(FINAL_ZIP_PART3) else 0.0
tot_w, usd_w, free_w = shutil.disk_usage(WORKING_DIR)

print("\n" + "=" * 80)
print("🎉 [HOÀN TẤT THÀNH CÔNG PART 3] PHÂN MẢNH GOLDEN DATASET PART 3 ĐÃ SẴN SÀNG!")
print(f"   📁 File thành phẩm Part 3     : {FINAL_ZIP_PART3}")
print(f"   📦 Kích thước file Part 3     : {p3_size_mb:.2f} MB")
print(f"   📊 Số Golden Triplets Part 3  : {len(meta_part3):,} clips")
print(f"   📑 File Triplets Index Part 3 : {CSV_PART3_PATH}")
print(f"   📑 File Audit Report 12 chỉ số: {AUDIT_PART3_PATH}")
print(f"   💾 Dung lượng đĩa còn trống    : {free_w/(1024**3):.2f} GB / {tot_w/(1024**3):.2f} GB")
print("=" * 80)
print("👉 HƯỚNG DẪN DÀNH CHO BẠN:")
print(f"   1. Nhìn sang panel Output bên phải giao diện Kaggle và TẢI FILE '{os.path.basename(FINAL_ZIP_PART3)}' VỀ MÁY.")
print(f"   2. TẢI CÁC FILE CSV '{os.path.basename(CSV_PART3_PATH)}' VÀ '{os.path.basename(AUDIT_PART3_PATH)}' VỀ MÁY.")
print("   3. Đặt các file tải về vào thư mục 'pipelines/dataset_builder/sakuga_pipeline v2/' trên máy tính.")
print("   4. Chạy script lắp ráp local (03_assemble_golden_dataset.py) để hợp nhất toàn bộ Part 1, 2 và 3 thành dataset cuối cùng!")
print("=" * 80)


---
### 🔍 Bước 9: (Tùy Chọn) Kiểm Tra Mẫu Golden Triplet Part 3 (Sanity Check)


In [ ]:
# Kiểm tra trực quan ngẫu nhiên 1 triplet trong Part 3
inspect_golden_zip_sample(FINAL_ZIP_PART3, "PART 3")
